# Set up


In [1]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
!pip install -U timesfm
import timesfm3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 4.3 MB/s eta 0:00:00


In [2]:
df = pd.read_csv('merged_energy_weather.csv')
df.columns.tolist()
df.head()

,date,target,building,mean_kw,max_kw,min_kw,n_points,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_min,precipitation_sum,rain_sum,snowfall_sum,windspeed_10m_max,windgusts_10m_max,winddirection_10m_dominant
0,2023-08-08,120MapleAve.Elec.PowerScout3HD/kW,120MapleAve,38.3820,77.395228,28.351630,7,68.5,63.1,66.4,73.2,63.3,0.157,0.157,0.0,11.5,26.2,265.0
1,2023-08-09,120MapleAve.Elec.PowerScout3HD/kW,120MapleAve,70.4929,131.581850,26.408410,24,78.7,61.5,69.5,81.0,62.5,0.008,0.008,0.0,9.6,23.0,282.0
2,2023-08-10,120MapleAve.Elec.PowerScout3HD/kW,120MapleAve,67.3543,112.166175,22.985750,24,75.4,62.2,66.6,78.6,64.2,0.756,0.756,0.0,9.9,22.1,198.0
3,2023-08-11,120MapleAve.Elec.PowerScout3HD/kW,120MapleAve,63.3860,114.478875,23.278665,24,77.7,61.1,67.5,76.9,63.3,0.024,0.024,0.0,9.1,22.4,300.0
4,2023-08-12,120MapleAve.Elec.PowerScout3HD/kW,120MapleAve,36.1651,78.608960,21.552210,24,81.7,59.6,70.3,85.3,60.0,0.268,0.268,0.0,9.8,25.1,179.0


In [3]:
config = timesfm3.ModelConfig(
    checkpoint_path="google/timesfm-3.0-pytorch",
    device="cpu"
)

forecaster = timesfm3.TimesFM3Evaluator(config)

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.32GB            

model.safetensors: downloading bytes:           |  0.00B            

# Createing Variables

In [4]:
context_len = 512 # window of past days to look at, max!
horizon = 14 # number of days its predicating
step_size = 14 # move forward the window
weather_cols = ['temperature_2m_max',
 'temperature_2m_min',
 'temperature_2m_mean',
 'apparent_temperature_max',
 'apparent_temperature_min',
 'precipitation_sum',
 'rain_sum',
 'snowfall_sum',
 'windspeed_10m_max',
 'windgusts_10m_max',
 'winddirection_10m_dominant']

buildings = ['AnabelTaylorHall','120MapleAve'] # only a couple to begin with


In [5]:
predictions = []
for b in buildings:
  b_df = df[df['building'] == b].sort_values('date').reset_index(drop=True) # orders them by date, 0 ,1 ,2. need for time series

  # move by the window
  for start_idx in range(0, len(b_df) - context_len - horizon + 1, step_size):
    print(start_idx)
    input = b_df['mean_kw'].values[start_idx : start_idx + context_len] # get the past 512 data
    weather_input = b_df[weather_cols].values[start_idx : start_idx + context_len + horizon].T.astype(np.float32)
    out = list(
            forecaster.predict_batch(
                contexts=[input],
                horizon=horizon,
                past_future_covariates=[weather_input],
                return_quantiles=False # change this to true for the confindnce levels when we are doing treaining
            )
        )
    pred_kw = out[0].forecast.squeeze()
    predictions.append(pred_kw)



0
14
28
42
56
70
84
98
112
126
140
154
168
182
196
210
224
238
252
266
280
294
308
322
336
350
364
378
392
406
420
434
448
462
476
490
504
518
532
546
560
574
588
602
616
630
644
658
672
686
700
714
728
742
756
770
784
798
812
826
840
854
868
882
896
910
924
938
952
966
980
994
1008
1022
1036
1050
1064
1078
1092
1106
1120
1134
1148
1162
1176
1190
1204
1218
1232
1246
1260
1274
1288
0
14
28
42
56
70
84
98
112
126
140
154
168
182
196
210
224
238
252
266
280
294
308
322
336
350
364
378
392
406
420
434
448
462
476
490
504
518
532
546
560
574
588
602


In [6]:
predictions

[array([17.591133, 17.795357, 18.259212, 18.051756, 17.331045, 17.682417,
        18.197166, 18.139849, 18.01718 , 18.682621, 18.172657, 17.543566,
        17.901161, 18.138252], dtype=float32),
 array([17.577606, 17.311905, 17.316381, 16.66352 , 16.216526, 16.710186,
        17.333546, 17.597183, 17.434301, 17.328945, 16.702124, 16.353573,
        16.726402, 17.375092], dtype=float32),
 array([18.243212, 18.245232, 18.032257, 17.547295, 17.462475, 17.53878 ,
        17.54012 , 17.722504, 17.43039 , 17.93179 , 17.228113, 16.662685,
        17.225866, 17.423578], dtype=float32),
 array([17.348906, 17.450308, 17.33086 , 16.136799, 16.422073, 16.473196,
        16.770908, 16.783138, 16.674614, 17.093922, 16.469234, 16.083172,
        16.248363, 16.392298], dtype=float32),
 array([16.683598, 16.220478, 16.127289, 15.324557, 14.942604, 15.624329,
        16.579891, 16.873947, 16.473637, 15.991116, 15.490608, 15.3666  ,
        15.920301, 16.479137], dtype=float32),
 array([16.39423 , 16.509